# Fine Tune LLM Tutorial

In [1]:
# import trl
# import datasets
# import accelerate
# import gradio as gr
import torch

In [2]:
MODEL_NAME = "google/gemma-3-270m-it"

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM


# MODEL_NAME = "unsloth/gpt-oss-20b"
max_sequence_length = 512

# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME,
#     dtype="auto",
#     device_map="auto",
#     attn_implementation="eager",
#     # load_in_4bit = True,
#     # full_finetuning = False,
#     # max_sequence_length=max_sequence_length,
# )


model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype="auto",
    device_map="auto",
    attn_implementation="eager",
    torch_dtype=torch.bfloat16)



/Users/carlos/Developer/llm_finetune_test/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0).
W0516 20:28:46.286000 10904 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Loading weights: 100%|██████████| 236/236 [00:00<00:00, 645.34it/s]


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"[INFO] Model on device: {model.device}")
print(f"[INFO] Model dtype: {model.dtype}")

[INFO] Model on device: mps:0
[INFO] Model dtype: torch.bfloat16


Our model requires numbers (tokens) as inputs.
We can turn strings into tokens via tokenizer!

In [5]:
tokenizer("hello world")

{'input_ids': [2, 23391, 1902], 'attention_mask': [1, 1, 1]}

In [6]:
import torch

outputs = model(torch.tensor(tokenizer("hello world")["input_ids"]).unsqueeze(0).to(model.device))
outputs.keys()

odict_keys(['logits', 'past_key_values'])

## Get dataset

In [7]:
from datasets import load_dataset

dataset = load_dataset("mrdbourke/FoodExtract-1k")

print(f"[INFO] Dataset keys: {dataset['train'].column_names}")
print(f"[INFO] Number of samples in the dataset: {len(dataset['train'])}")

[INFO] Dataset keys: ['sequence', 'image_url', 'class_label', 'source', 'char_len', 'word_count', 'syn_or_real', 'uuid', 'gpt-oss-120b-label', 'gpt-oss-120b-label-condensed', 'target_food_names_to_use', 'caption_detail_level', 'num_foods', 'target_image_point_of_view']
[INFO] Number of samples in the dataset: 1420


## Analyze dataset

In [8]:
import json
import random

def get_random_idx(dataset):
    return_idx = random.randint(0, len(dataset) - 1)
    return return_idx  

In [9]:


random_idx = get_random_idx(dataset['train'])
random_sample = dataset['train'][random_idx]

example_input = random_sample['sequence']
example_output = random_sample['gpt-oss-120b-label']
example_output_condensed = random_sample['gpt-oss-120b-label-condensed']

print(f"[INFO] Example input: {example_input}")
print()
print(f"[INFO] Example output: {example_output}")
print()
print(f"[INFO] Example output condensed: {example_output_condensed}")


[INFO] Example input: This image shows the back of a food product box with detailed information. The product is labeled as "LUNCHBOX FRIENDLY NATURAL COLOURS & FLAVOURS." The ingredients list includes Cornflakes (32%) (Corn, Sugar, Salt), Honey (17%), Rice Bran Puffs (17%) (Rice Flour, Rice Bran), Tapioca Fibre, Canola Oil, Corn Fibre, and Natural Flavours. The product may contain egg, milk, peanut, soy, and tree nuts. It contains 4.2g of sugar per teaspoon and should be stored in a cool, dry place. The brand name "Kez's Kitchen" is prominently displayed, with a note that "Kez's Kitchen and KEZ'S are Trade Marks of Mickies Bikkies Pty Ltd t/a Kez's Kitchen."

[INFO] Example output: {'is_food_or_drink': True, 'tags': ['il', 'fi', 'fp'], 'food_items': ['Cornflakes', 'Honey', 'Rice Bran Puffs', 'Tapioca Fibre', 'Canola Oil', 'Corn Fibre', 'Natural Flavours', 'egg', 'milk', 'peanut', 'soy', 'tree nuts', 'Corn', 'Sugar', 'Salt', 'Rice Flour', 'Rice Bran'], 'drink_items': []}

[INFO] Example

Because we'd like to use our model to potentially filter a large corpus of data, we get it to assign various tags to the text as well.

In [10]:
tags_dict = {
    'il': 'ingredient_list',
    'me': 'menu',
    're': 'recipe',
    'fi': 'food_item',
    'di': 'drink_item',
    'fa': 'food_advertisement',
    'fp': 'food_packaging',
}

## Format the dataset into LLM-style inputs/outputs
Right now we have examples of string-based inputs and structured outputs.

however, our LLMs generally want things in the format of:

```{"user": "Hello my name is Carlos", "assistant": "I'm an LLM"}```

In [11]:
def sample_to_conversation(sample):
    """Helper function to convert a sample from the dataset into a conversation format for fine-tuning."""
    return {
        "messages": [
            {"role": "user", "content": sample['sequence']},
            {"role": "system", "content": sample['gpt-oss-120b-label-condensed']}
        ]
    }

sample_to_conversation(random_sample)

{'messages': [{'role': 'user',
   'content': 'This image shows the back of a food product box with detailed information. The product is labeled as "LUNCHBOX FRIENDLY NATURAL COLOURS & FLAVOURS." The ingredients list includes Cornflakes (32%) (Corn, Sugar, Salt), Honey (17%), Rice Bran Puffs (17%) (Rice Flour, Rice Bran), Tapioca Fibre, Canola Oil, Corn Fibre, and Natural Flavours. The product may contain egg, milk, peanut, soy, and tree nuts. It contains 4.2g of sugar per teaspoon and should be stored in a cool, dry place. The brand name "Kez\'s Kitchen" is prominently displayed, with a note that "Kez\'s Kitchen and KEZ\'S are Trade Marks of Mickies Bikkies Pty Ltd t/a Kez\'s Kitchen."'},
  {'role': 'system',
   'content': 'food_or_drink: 1\ntags: il, fi, fp\nfoods: Cornflakes, Honey, Rice Bran Puffs, Tapioca Fibre, Canola Oil, Corn Fibre, Natural Flavours, egg, milk, peanut, soy, tree nuts, Corn, Sugar, Salt, Rice Flour, Rice Bran\ndrinks:'}]}

In [12]:
# Map our sample_to_conversation function across the entire dataset
dataset = dataset.map(sample_to_conversation, batched=False)

dataset['train'][0]

{'sequence': 'A mouth-watering photograph captures a delectable dish centered on a rectangular white porcelain plate, resting on a rustic wooden tabletop indoors. In the background, a wooden cutting board with a long handle subtly enhances the setting. The plate is adorned with several generously-sized, cheese-stuffed peppers that have been roasted to perfection, their blistered skins marked by charred black spots. Split down the middle, the peppers reveal a creamy white cheese filling, enriched with a blend of aromatic herbs. Once stuffed, the peppers have been closed and roasted, achieving a luscious, smoky flavor.\n\nThe dish is elegantly garnished with vibrant cherry tomato halves, freshly chopped green herbs, and delicate sprinkles of small diced red onions. A light, possibly citrus-infused dressing, hinted by a sheen of oil or lime juice, gently coats the ensemble, adding an extra layer of freshness. The meticulous presentation and vivid colors make this image not only a feast fo

Notice how we now have a ```"messages"``` key in our dataset samples.

In [13]:
dataset = dataset["train"].train_test_split(test_size=0.2,
                                            shuffle=False,
                                            seed=42)

dataset

DatasetDict({
    train: Dataset({
        features: ['sequence', 'image_url', 'class_label', 'source', 'char_len', 'word_count', 'syn_or_real', 'uuid', 'gpt-oss-120b-label', 'gpt-oss-120b-label-condensed', 'target_food_names_to_use', 'caption_detail_level', 'num_foods', 'target_image_point_of_view', 'messages'],
        num_rows: 1136
    })
    test: Dataset({
        features: ['sequence', 'image_url', 'class_label', 'source', 'char_len', 'word_count', 'syn_or_real', 'uuid', 'gpt-oss-120b-label', 'gpt-oss-120b-label-condensed', 'target_food_names_to_use', 'caption_detail_level', 'num_foods', 'target_image_point_of_view', 'messages'],
        num_rows: 284
    })
})

## Try the model with a pipeline

In [14]:
easy_sample = {"role": "user", "content": "Hi my name is Carlos"}

def create_easy_sample(input):
    template = {"role": "user", "content": input}
    return template


In [15]:
from transformers import pipeline

# Load model and use it as a pipeline
pipe = pipeline("text-generation",
                model=model,
                tokenizer=tokenizer)

input_text = "My name is Carlos. Please reply to me with a machine learning poem"
easy_sample = create_easy_sample(input_text)
input_prompt = pipe.tokenizer.apply_chat_template([easy_sample], tokenize=False, add_generation_prompt=True)

print(f"[INFO] Input prompt: {input_prompt}")

default_outputs = pipe(input_prompt,
                       max_new_tokens=512,
                       disable_compile=True)

print(f"[INFO] Input text: {input_text}")
print()
print(f"[INFO] Output from {MODEL_NAME}:")
print()
print(f"[INFO] Model output: {default_outputs[0]['generated_text'][len(input_prompt):]}")


Passing `generation_config` together with generation-related arguments=({'disable_compile', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[INFO] Input prompt: <bos><start_of_turn>user
My name is Carlos. Please reply to me with a machine learning poem<end_of_turn>
<start_of_turn>model

[INFO] Input text: My name is Carlos. Please reply to me with a machine learning poem

[INFO] Output from google/gemma-3-270m-it:

[INFO] Model output: Okay, here's a machine learning poem for Carlos. I've tried to make it engaging and evocative, focusing on the feeling of connection and the potential for learning:

**Carlos's Algorithm**

A seed of data, a silent hum,
A symphony of learning to come.
From data streams, a nascent grace,
A learning engine, finding its place.

Binary code, a tangled thread,
Unraveling patterns, putting hope to tread.
Each point a journey, a whispered plea,
To build a future, for you and me.

The algorithms learn and grow,
With every step, a vibrant flow.
A tapestry of knowledge bright,
Reflecting beauty, day and night.

Though fear may linger, a subtle sting,
A flicker of understanding wing.
For in this code, 

## Try the model on one of our sequences

In [16]:
# Get a random sample
random_idx = get_random_idx(dataset['train'])
random_train_sample = dataset['train'][random_idx]

# apply the chat template
input_prompt = pipe.tokenizer.apply_chat_template(conversation=random_train_sample["messages"][:1], tokenize=False, add_generation_prompt=True)

# let's run the default model on our input
default_outputs = pipe(input_prompt,
                       max_new_tokens=256)

# View the results
print(f"[INFO] Input prompt: {input_prompt}")
print()
print(f"[INFO] Model output: {default_outputs[0]['generated_text'][len(input_prompt):]}")

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[INFO] Input prompt: <bos><start_of_turn>user
The image shows a package of SunRice® Original Thick Rice Cakes. The nutrition information panel is prominently displayed, detailing the following:

- Servings per package: 6
- Serving size: 25 g

Nutrition Information per Serving:
- Energy: 408 kJ (97 kcal)
- Protein: 2.0 g
- Gluten: 0 mg
- Fat, total: 0.8 g
  - Saturated: 0.2 g
  - Trans: 0 g
- Cholesterol: 0 mg
- Carbohydrate: 19.6 g
  - Sugars: 0.3 g
- Dietary fibre: 1.1 g
- Sodium: 0.3 mg

Nutrition Information per 100 g:
- Energy: 1630 kJ (389 kcal)
- Protein: 8.1 g
- Gluten: 0 mg
- Fat, total: 3.3 g
  - Saturated: 0.7 g
  - Trans: 0 g
- Cholesterol: 0 mg
- Carbohydrate: 78.5 g
  - Sugars: 1.2 g
- Dietary fibre: 4.4 g
- Sodium: 1.0 mg

Ingredients: Whole grain brown rice.

The package also includes storage instructions: "Store in an airtight container to maintain freshness, in a cool, dry place out of sunlight."

Additional details:
- Made in Australia from 100% Australian ingredients

By default the model produces a fairly generic response, this is expected and good.  It means the model has a good understanding of langugage.

## Let's try and prompt the model

In [17]:
prompt_instruction = """Given the following target input text from an image caption, please extract the food and drink items to a list.
If there are no food or drink items, return an empty list.

Return in the following format:
food_items: [item_1, item_2, item_3]
drink_items: [item_4, item_5]

For exmaple:
Input text: hello my name is carlos.
Output:
food_items: []
drink_items: []

Example 2:
Input text: A plate of rice cakes, salmon, cottage cheese and small cherry tomatoes with a cup of tea.
Output:
food_items: ['rice cakes', 'salmon', 'cottage cheese', 'cherry tomatoes']
drink_items: ['cup of tea']

return only the formatted output and nothing else.
target input text: <targ_input_text>"""

def update_input_message_content(input):
    original_content = input["messages"][:1][0]["content"]
    new_content = prompt_instruction.replace("<targ_input_text>", original_content)

    new_input = [{"content": new_content, "role": "user"}]
    return new_input

print(f"[INFO] Original input message content: {random_train_sample['messages'][:1][0]['content']}")
print()
print(f"[INFO] new content with instructions in prompt:")
print(update_input_message_content(input=random_train_sample)[0]["content"])

[INFO] Original input message content: The image shows a package of SunRice® Original Thick Rice Cakes. The nutrition information panel is prominently displayed, detailing the following:

- Servings per package: 6
- Serving size: 25 g

Nutrition Information per Serving:
- Energy: 408 kJ (97 kcal)
- Protein: 2.0 g
- Gluten: 0 mg
- Fat, total: 0.8 g
  - Saturated: 0.2 g
  - Trans: 0 g
- Cholesterol: 0 mg
- Carbohydrate: 19.6 g
  - Sugars: 0.3 g
- Dietary fibre: 1.1 g
- Sodium: 0.3 mg

Nutrition Information per 100 g:
- Energy: 1630 kJ (389 kcal)
- Protein: 8.1 g
- Gluten: 0 mg
- Fat, total: 3.3 g
  - Saturated: 0.7 g
  - Trans: 0 g
- Cholesterol: 0 mg
- Carbohydrate: 78.5 g
  - Sugars: 1.2 g
- Dietary fibre: 4.4 g
- Sodium: 1.0 mg

Ingredients: Whole grain brown rice.

The package also includes storage instructions: "Store in an airtight container to maintain freshness, in a cool, dry place out of sunlight."

Additional details:
- Made in Australia from 100% Australian ingredients.
- Con

In [18]:
# apply the chat template
updated_input_prompt = update_input_message_content(input=random_train_sample)

input_prompt = pipe.tokenizer.apply_chat_template(conversation=updated_input_prompt, tokenize=False, add_generation_prompt=True)

# let's run the default model on our input
default_outputs = pipe(text_inputs=input_prompt,
                       max_new_tokens=256,)

# view and compare the outputs
print(f"[INFO] Input prompt: {input_prompt}")
print(f"[INFO] Default outputs: {default_outputs[0]['generated_text'][len(input_prompt):]}")


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[INFO] Input prompt: <bos><start_of_turn>user
Given the following target input text from an image caption, please extract the food and drink items to a list.
If there are no food or drink items, return an empty list.

Return in the following format:
food_items: [item_1, item_2, item_3]
drink_items: [item_4, item_5]

For exmaple:
Input text: hello my name is carlos.
Output:
food_items: []
drink_items: []

Example 2:
Input text: A plate of rice cakes, salmon, cottage cheese and small cherry tomatoes with a cup of tea.
Output:
food_items: ['rice cakes', 'salmon', 'cottage cheese', 'cherry tomatoes']
drink_items: ['cup of tea']

return only the formatted output and nothing else.
target input text: The image shows a package of SunRice® Original Thick Rice Cakes. The nutrition information panel is prominently displayed, detailing the following:

- Servings per package: 6
- Serving size: 25 g

Nutrition Information per Serving:
- Energy: 408 kJ (97 kcal)
- Protein: 2.0 g
- Gluten: 0 mg
- Fat,

In [19]:
# this is our input
print(random_train_sample['messages'][0]['content'])
print()

# this is our ideal output
print(random_train_sample['messages'][1]['content'])

The image shows a package of SunRice® Original Thick Rice Cakes. The nutrition information panel is prominently displayed, detailing the following:

- Servings per package: 6
- Serving size: 25 g

Nutrition Information per Serving:
- Energy: 408 kJ (97 kcal)
- Protein: 2.0 g
- Gluten: 0 mg
- Fat, total: 0.8 g
  - Saturated: 0.2 g
  - Trans: 0 g
- Cholesterol: 0 mg
- Carbohydrate: 19.6 g
  - Sugars: 0.3 g
- Dietary fibre: 1.1 g
- Sodium: 0.3 mg

Nutrition Information per 100 g:
- Energy: 1630 kJ (389 kcal)
- Protein: 8.1 g
- Gluten: 0 mg
- Fat, total: 3.3 g
  - Saturated: 0.7 g
  - Trans: 0 g
- Cholesterol: 0 mg
- Carbohydrate: 78.5 g
  - Sugars: 1.2 g
- Dietary fibre: 4.4 g
- Sodium: 1.0 mg

Ingredients: Whole grain brown rice.

The package also includes storage instructions: "Store in an airtight container to maintain freshness, in a cool, dry place out of sunlight."

Additional details:
- Made in Australia from 100% Australian ingredients.
- Contact information: 1800 255 999 Australi

Our LLM doesn't do exactly what we want, but that's ok, we'll fine tune it to get it to do exactly what we want.

## Fine-tuning our model
Steps:
 1. Setup SFTConfig (Supervised fine tuning)
 2. Use SFTTrainer to train our model on our supervised samples

In [20]:
CHECKPOINT_DIR_NAME = "./checkpoint_models"
BASE_LEARNING_RATE = 2e-5

In [21]:
# setting up our SFTConfig

from trl import SFTConfig

torch_dtype = model.dtype



print(f"[INFO] Torch dtype: {torch_dtype}")
print(f"[INFO] Base learning rate: {BASE_LEARNING_RATE}")

# sft_config = SFTConfig(
#     output_dir=CHECKPOINT_DIR_NAME,
#     max_length=max_sequence_length,
#     packing=False,
#     num_train_epochs=3,
#     per_device_train_batch_size=4,
#     per_device_eval_batch_size=4,
#     eval_accumulation_steps=1,
#     gradient_checkpointing=True,
#     optim="adamw_torch", #"adamw_torch_fused",
#     logging_steps=1,
#     save_strategy="epoch",
#     eval_strategy="epoch",
#     learning_rate=BASE_LEARNING_RATE,
#     fp16=False, #True if torch_dtype == torch.float16 else False,
#     bf16=True, #if torch_dtype == torch.bfloat16 else False,
#     lr_scheduler_type="constant",
#     push_to_hub=False,
#     report_to=[],
#     max_grad_norm=0.3,
# )

sft_config = SFTConfig(
    output_dir=CHECKPOINT_DIR_NAME,
    
    # --- Data Processing Parameters ---
    max_length=256,                     # Strictly bound the sequence size here
    packing=True,                       # Packs short sequences to stabilize the MPS allocator
    dataset_text_field="text",          # Change to your dataset text column name
    
    # --- Core Training Parameters ---
    per_device_train_batch_size=2,      # Small batch size to prevent fragmentation
    gradient_accumulation_steps=4,      # Simulates an effective batch size of 8
    learning_rate=2e-5,
    optim="adamw_torch",                # Standard torch optimizer for MPS stability
    
    # --- Crucial Memory & MPS Stability Fixes ---
    gradient_checkpointing=True,        # Clears activation tensors on forward pass
    gradient_checkpointing_kwargs={"use_reentrant": False}, # Vital for PyTorch Mac stability
    bf16=True,                          # Matches native Mac architecture
    fp16=False,                         # Set to False; mixed precision fp16 causes leaks on Mac
    
    # --- Tracking & Eval Controls ---
    eval_strategy="no",                 # Turn off evaluation to isolate memory leaks
    report_to="none"
)

[INFO] Torch dtype: torch.bfloat16
[INFO] Base learning rate: 2e-05


In [22]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    args=sft_config,
    processing_class=tokenizer
)

# trainer.train()

[RANK 0] Padding-free training is enabled, but the attention implementation is not set to a supported flash attention variant. Padding-free training flattens batches into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn, kernels-community/flash-attn3, kernels-community/vllm-flash-attn3. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation` in the model configuration to one of these supported options or verify that your attention mechanism can handle flattened sequences.
[RANK 0] You are using packing, but the attention implementation is not set to a supported flash attention variant. Packing gathers multiple samples into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn, kernels-community/flash-attn3, kernels-

In [23]:
# trainer.save_model()

In [24]:
# Load the fine-tuned model for inference
from transformers import AutoTokenizer, AutoModelForCausalLM

loaded_model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=CHECKPOINT_DIR_NAME,
    dtype="auto",
    device_map="auto",
    attn_implementation="eager",
    )

Loading weights: 100%|██████████| 236/236 [00:00<00:00, 797.78it/s]


In [25]:
from transformers import pipeline

# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

loaded_model_pipeline = pipeline("text-generation",
    model=loaded_model,
    tokenizer=tokenizer)

loaded_model_pipeline


TextGenerationPipeline: {'model': 'Gemma3ForCausalLM', 'dtype': 'bfloat16', 'device': 'mps', 'input_modalities': ('image', 'text'), 'output_modalities': ('text',)}

In [26]:
dataset["test"]

Dataset({
    features: ['sequence', 'image_url', 'class_label', 'source', 'char_len', 'word_count', 'syn_or_real', 'uuid', 'gpt-oss-120b-label', 'gpt-oss-120b-label-condensed', 'target_food_names_to_use', 'caption_detail_level', 'num_foods', 'target_image_point_of_view', 'messages'],
    num_rows: 284
})

In [30]:
# get a. random sample
random_test_idx = get_random_idx(dataset['test'])
random_test_sample = dataset['test'][random_test_idx]

# apply the chat template
input_prompt = pipe.tokenizer.apply_chat_template(conversation=random_test_sample["messages"][:1], tokenize=False, add_generation_prompt=True)  

# let's run the default model on our input
default_outputs = loaded_model_pipeline(text_inputs=input_prompt,
                                        max_new_tokens=256)

# view and compare the outputs
print(f"[INFO] Input prompt: {input_prompt}")
print(f"[INFO] Default outputs: {default_outputs[0]['generated_text'][len(input_prompt):]}")


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[INFO] Input prompt: <bos><start_of_turn>user
police stand guard in the background with women wearing pink hair and carrying rainbow flags in the foreground<end_of_turn>
<start_of_turn>model

[INFO] Default outputs: food_or_drink: 0
tags: 
foods: 
drinks: 



In [31]:
def get_model_num_params(model):
    """returns the number of trainable, non-trainable and total parameters in the model."""
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_trainable_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    total_params = trainable_params + non_trainable_params
    return {
        "trainable_params": trainable_params,
        "non_trainable_params": non_trainable_params,
        "total_params": total_params
    }

model_params = get_model_num_params(loaded_model)
print(f"[INFO] Trainable parameters: {model_params['trainable_params']}")
print(f"[INFO] Non-trainable parameters: {model_params['non_trainable_params']}")
print(f"[INFO] Total parameters: {model_params['total_params']}")


[INFO] Trainable parameters: 268098176
[INFO] Non-trainable parameters: 0
[INFO] Total parameters: 268098176


In [34]:
# our model is 270 million parameters, gpt-oss-120b is 120 billion parameters, so our model is about 0.225% of the size of gpt-oss-120b. It's quite a bit smaller!
print(120_000_000_000 / 270_000_000)
print(270_000_000 / 120_000_000_000)

444.44444444444446
0.00225


## Export model

In [40]:
# At the end of your training script:
trainer.save_model("./food_extract_fine_tuned_model")
tokenizer.save_pretrained("./food_extract_fine_tuned_model")


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.07it/s]


('./food_extract_fine_tuned_model/tokenizer_config.json',
 './food_extract_fine_tuned_model/chat_template.jinja',
 './food_extract_fine_tuned_model/tokenizer.json')

In [41]:
tokenizer.vocab_size



262144